In [1]:
from spin_lattices import KagomeLattice, SpinLattice
from heisenberg_hamiltonians import HeisenbergJ1J2, SpinSystem
from boolean_analysis import (
    BooleanFourierAnalyzer,
    keep_largest_n,
    keep_everything,
    ScorerType,
    get_scorer,
    SignalOption,
    AmplitudeMedianBinSignalKind,
    SignSignalKind,
    AmplitudeSignalKind,
    SignalKind,
)
from boolean_fourier_learner import BooleanFourierLearner

from pathlib import Path
import numpy as np
import pandas as pd
import lattice_symmetries as ls
import matplotlib.pyplot as plt
from heisenberg_hamiltonians import batched_state_info_df
from itertools import product
import numpy.typing as npt
from tqdm import tqdm
import seaborn as sns
import parse

from parity import popcount, parity

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
from misc_utils import make_unpacked_configurations

from pytorchtools import EarlyStopping

import pickle
from spin_nn import SpinNN
import datetime

ground_state_cache_dir = Path("groundstates")
fourier_learners_cache_dir = Path("fourier_learners_cache")
experiments_dir = Path("experiments") / "kagome-24-nn-2023-01-27"
experiments_dir.mkdir(parents=True, exist_ok=True)

2023-01-30 19:31:16.618 | DEBUG    | lattice_symmetries:__init__:49 - Initializing Haskell runtime...
2023-01-30 19:31:16.620 | DEBUG    | lattice_symmetries:__init__:51 - Initializing Chapel runtime...
[Debug]   [2023-01-30 19:31:16.674 | DEBUG    | lattice_symmetries:__init__:53 - Setting Python exception handler...
LOCALE0]   Initializing chpl_kernels ...
set_python_exception_handler ...
/vol/tcm10/ischurov/frustrations-eda/boolean_analysis.py:28: UserWarning: This module is deprecated and will be removed in the future.
  warnings.warn("This module is deprecated and will be removed in the future.")
/vol/tcm10/ischurov/frustrations-eda/boolean_analysis.py:29: UserWarning: Use lattice_boolean_analysis.py instead.
  warnings.warn("Use lattice_boolean_analysis.py instead.")


In [2]:
class FC1SpinNN(SpinNN):
    def __init__(self, lattice: SpinLattice, hidden_size: int):
        super().__init__(lattice)
        input_size = lattice.number_spins
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, 2)

    def forward(self, inp: torch.Tensor) -> torch.Tensor:
        x = self.preprocess(inp)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return  self.postprocess(x)
        

In [3]:
J2 = 0.8
system = HeisenbergJ1J2(
    lattice=KagomeLattice(width=2, height=4),
    J1=1,
    J2=J2,
    use_symmetries=True,
    spin_inversion=1,
    ground_state_cache_dir=ground_state_cache_dir,
    show_progress=True,
)
system.get_eigenstates(1)

number_spins=24
Symmetry group contains 16 elements
Hilbert space dimension is 85662
Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-0.8-True-1-1.pickle
Ground state energy is -40.5183420067


[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


(array([-40.51834201]),
 array([[ 8.36163038e-08],
        [ 1.13414544e-07],
        [ 1.79331566e-08],
        ...,
        [-8.66655973e-03],
        [-1.17282633e-02],
        [ 3.99580256e-03]]))

In [4]:
df = (
    system.get_df_ground_state(
        canonical_basis=True,
    )
    .assign(
        sign=(lambda df: np.sign(df["eigenstate_coeff"])),
        prob=(lambda df: np.abs(df["eigenstate_coeff"]) ** 2),
    )
    .assign(y=lambda df: (df["sign"] == 1).astype(int))
)

In [5]:
df_rep = df.join(system.lattice.get_state_info_df(hamming_weight=system.lattice.number_spins // 2)).groupby('representative').agg({'sign': 'mean', 'prob': 'sum', 'y': 'mean'})

[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...


In [18]:
eps_train = 1e-3
val_eps = 1e-2
test_eps = 1e-2
batch_size = 64

df_train = df_rep.sample(frac=eps_train, weights="prob")
df_rep_for_val = df_rep.drop(df_train.index)
df_val = df_rep_for_val.sample(frac=val_eps, weights="prob")
df_rep_for_test = df_rep_for_val.drop(df_val.index)
df_test = df_rep_for_test.sample(frac=test_eps, weights="prob")

n_batches = int(np.ceil(len(df_train) / batch_size))
epochs = 20000


In [19]:
df_train

,sign,prob,y
representative,,,
5679818,1.0,0.000196,1.0
3324238,-1.0,0.001202,0.0
3380563,1.0,0.000308,1.0
3618118,-1.0,0.000526,0.0
1496522,1.0,0.001294,1.0
...,...,...,...
3586402,1.0,0.000167,1.0
3569994,-1.0,0.000886,0.0
1404588,-1.0,0.000049,0.0


In [20]:
net = FC1SpinNN(lattice=system.lattice, hidden_size=64)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)


In [21]:
def get_inputs_and_labels(df: pd.DataFrame) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    X = torch.tensor(
        make_unpacked_configurations(np.asarray(df.index, dtype="uint64"), number_spins=system.lattice.number_spins).astype('float32')
    )
    y = torch.tensor(df["y"].values.astype("int8"), dtype=torch.long)
    probs = torch.tensor(df["prob"].values.astype("float"), dtype=torch.float32)
    return X, y, probs

inputs_val, labels_val, probs_val = get_inputs_and_labels(df_val)


In [22]:
def evaluate(net, inputs, labels, probs):
    with torch.no_grad():
        outputs = net(inputs)
        _, predicted = torch.max(outputs.data, 1)
        correct = (predicted == labels).sum().item()
        accuracy = correct / len(labels)

        sign_overlap = (
            ((1 - 2 * predicted) * (1 - 2 * labels) * probs).sum() / probs.sum()
        ).item()
        return accuracy, sign_overlap


In [23]:
out = net(
    torch.Tensor(
        make_unpacked_configurations(
            np.asarray(
                net.lattice.get_state_info_df(hamming_weight=net.lattice.number_spins // 2, spin_inversion=1)
                .query("representative == 23550")
                .index
            ),
            net.lattice.number_spins,
        ).astype("float32")
    )
)
assert torch.isclose(out, out[0]).all()


In [24]:
out

tensor([[-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569],
        [-0.0466, -0.0569]], grad_fn=<MeanBackward1>)

In [25]:
early_stopping = EarlyStopping(patience=1000, delta=0.01, verbose=True)

In [26]:
for epoch in range(epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    i = None
    loss = None

    net.train()
    for i in range(n_batches):
        data = df_train.iloc[i * batch_size : (i + 1) * batch_size]
        inputs, labels, probs = get_inputs_and_labels(data)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()
    net.eval()
    print(f"[{epoch + 1}, {i}] loss: {loss}")

    accuracy, sign_overlap = evaluate(net, inputs, labels, probs)
    print(f"Test set: accuracy: {100 * accuracy} %, sign overlap: {sign_overlap}")

    accuracy_val, sign_overlap_val = evaluate(net, inputs_val, labels_val, probs_val)
    print(f"Validation set: accuracy: {100 * accuracy_val} %, sign overlap: {sign_overlap_val}")
    
    early_stopping(-sign_overlap_val, net)
    if early_stopping.early_stop:
        print("Early stopping")
        break
    
net.load_state_dict(torch.load('checkpoint.pt'))

    


[1, 2] loss: 0.7017393708229065
Test set: accuracy: 45.23809523809524 %, sign overlap: 0.056498508900403976
Validation set: accuracy: 50.794585050029426 %, sign overlap: -0.020984522998332977
Validation loss decreased (inf --> 0.020985).  Saving model ...
[2, 2] loss: 0.7036336064338684
Test set: accuracy: 45.23809523809524 %, sign overlap: 0.056498508900403976
Validation set: accuracy: 50.794585050029426 %, sign overlap: -0.020984522998332977
EarlyStopping counter: 1 out of 1000
[3, 2] loss: 0.7039008140563965
Test set: accuracy: 45.23809523809524 %, sign overlap: 0.056498508900403976
Validation set: accuracy: 50.794585050029426 %, sign overlap: -0.020984522998332977
EarlyStopping counter: 2 out of 1000
[4, 2] loss: 0.7033004760742188
Test set: accuracy: 45.23809523809524 %, sign overlap: 0.056498508900403976
Validation set: accuracy: 50.794585050029426 %, sign overlap: -0.020984522998332977
EarlyStopping counter: 3 out of 1000
[5, 2] loss: 0.7023251056671143
Test set: accuracy: 45.23

<All keys matched successfully>

In [27]:
evaluate(net, inputs_val, labels_val, probs_val)

(0.5726898175397293, 0.3428715467453003)

In [28]:
evaluate(net, *get_inputs_and_labels(df_test))

(0.5570749108204518, 0.19144122302532196)